# Experimental: the `coffea.compute` execution refactor

> ⚠️ **Experimental — more speculative than the previews in `coffea-05-preview.ipynb`.**
> This material tracks draft PR #1470, a large redesign of *how coffea executes work*. It is becoming likelier to land, but the API **will** change and the execution engine is a work in progress — several cells below only illustrate the intended shape rather than fully running.
>
> Guarded so it is safe to "Run All" in the default environment (cells print a note when the feature is absent). To run the live parts, use the experimental environment:
>
> ```bash
> pixi run -e experimental jupyter lab
> ```

## Why `coffea.compute`?

Today coffea executes analyses through the `processor.Runner` + `Executor` (`Iterative`/`Futures`/`Dask`) machinery from notebooks 03–04. PR #1470 proposes replacing that trio with a single `Backend` **protocol**:

- a **`Computable`** describes the work (a `Dataset` mapped through a function via `.map_steps`),
- any **`Backend`** runs it via `.compute(...)`, returning a non-blocking **`Task`**,
- the `Task` exposes `.result()`, `.partial_result()`, `.wait()`, `.status()`, `.cancel()` — enabling **resumable** jobs and **partial results** that the current `Runner` cannot offer.

This is a bigger, more experimental change than the dataset-tools previews in notebook 05, so treat the API as provisional.

In [ ]:
# Feature detection (never by version number). coffea.compute only exists on the
# PR #1470 branch installed by the `experimental` environment.
import importlib.util
from pathlib import Path

import coffea

HAS_COMPUTE = importlib.util.find_spec("coffea.compute") is not None  # draft PR #1470


def experimental_note(feature="coffea.compute", pr="#1470"):
    print(
        f"[experimental] '{feature}' is not in this coffea build ({coffea.__version__}).\n"
        f"               It lives in draft PR {pr}. Launch the experimental environment:\n"
        f"                 pixi run -e experimental jupyter lab"
    )


_demo_file = Path("../columnar/data/SMHiggsToZZTo4L.root")
print(f"coffea {coffea.__version__}")
print(f"  HAS_COMPUTE = {HAS_COMPUTE}")

## The building blocks

`coffea.compute` is organised into a few small pieces:

- **`coffea.compute.data`** — `Dataset`, `File`, `DataGroup` and their typed-metadata companions (`ContextDataset`, ...). `Dataset.map_steps(fn)` / `map_files(fn)` turn data + a function into a `Computable`.
- **`coffea.compute.protocol`** — the `Backend` / `RunningBackend` protocol, `Computable`, `WorkElement`, `Task`, the `TaskStatus` enum, and result-merging helpers (`ResultWrapper`, `Addable`).
- **`coffea.compute.backends`** — concrete backends; `ThreadedBackend` is the reference one.
- **`coffea.compute.errors`** — `ErrorPolicy` and friends for per-work-element failure handling.

The next cell lists whatever is actually present in your installed build.

In [ ]:
if HAS_COMPUTE:
    import coffea.compute as C
    from coffea.compute import protocol, data, errors
    from coffea.compute.backends import threaded

    def _classes(m):
        return [n for n in dir(m) if n[:1].isupper() and not n.startswith("_")]

    print("coffea.compute exposes:", C.__all__)
    print("protocol :", _classes(protocol))
    print("data     :", _classes(data))
    print("errors   :", _classes(errors))
    print("backends :", [n for n in _classes(threaded) if n.endswith("Backend")])
else:
    experimental_note()

## A first computation

Build a `Dataset` of `File`s (each with explicit `(start, stop)` steps), map a plain function over the steps to get a `Computable`, then hand it to a `Backend`. The target is a single, backend-agnostic one-liner. The execution engine is still WIP, so this is guarded and wrapped — the point is the *shape*.

In [ ]:
if HAS_COMPUTE:
    from coffea.compute.data import Dataset, File, ContextDataset
    from coffea.compute.backends.threaded import ThreadedBackend

    dataset = Dataset(
        files=[File(path=str(_demo_file), steps=[(0, 50_000), (50_000, 100_000)])],
        metadata=ContextDataset(dataset_name="demo", cross_section=None),
    )

    def process(events):          # a plain callable *is* the processor
        return len(events)

    computable = dataset.map_steps(process)
    n_files = len(dataset.files)
    n_steps = sum(len(f.steps) for f in dataset.files)
    print(f"built a Computable over {n_files} file(s) / {n_steps} step(s)")
    print("target one-liner:")
    print("    with ThreadedBackend() as backend:")
    print("        total = backend.compute(computable).result()")
    try:
        with ThreadedBackend() as backend:
            total = backend.compute(computable).result()
        print("result:", total)
    except Exception as exc:  # execution engine is a work in progress
        print(f"[experimental] execution did not complete here ({type(exc).__name__}); "
              "the protocol shape above is the point of this preview.")
else:
    experimental_note()

## Error handling and resumable tasks

A `Task` is non-blocking. You can poll `.status()`, fetch a `.partial_result()`, `.wait()`, or `.cancel()`, and an `ErrorPolicy` decides what happens when individual work elements fail (retry, skip, or abort) — so a job can resume the un-run remainder instead of starting over.

In [ ]:
if HAS_COMPUTE:
    import inspect
    from coffea.compute.errors import ErrorPolicy
    from coffea.compute.protocol import TaskStatus

    print("TaskStatus:", [s.name for s in TaskStatus])
    print("ErrorPolicy fields:", list(inspect.signature(ErrorPolicy).parameters))
    print()
    print("Intended resume pattern:")
    print("    policy = ErrorPolicy(continue_on=(OSError,))")
    print("    with ThreadedBackend() as backend:")
    print("        task = backend.compute(computable, error_policy=policy)")
    print("        task.wait()")
    print("        done, resumable = task.partial_result()   # partial + un-run remainder")
    print("        total = done + backend.compute(resumable).result()")
else:
    experimental_note()

## How today's executors map onto the protocol

| today (`coffea.processor`) | `coffea.compute` (PR #1470) |
|---|---|
| `ProcessorABC.process(events)` | any callable `process(events)` |
| `Runner` + `IterativeExecutor` | `ThreadedBackend()` / a serial backend |
| `Runner` + `FuturesExecutor` | a process-pool `Backend` |
| `Runner` + `DaskExecutor` | a distributed `Backend` |
| `fileset` dict + `run.preprocess` | `Dataset`/`File` + `.map_steps(process)` → `Computable` |
| blocking `out = run(...)` | non-blocking `task = backend.compute(...)`; then `task.result()` |

The goal: moving work from a laptop to many cores to a cluster becomes a one-line change of `Backend`, with the *same* `Computable` — plus resumable `Task`s and partial results the current `Runner` cannot provide.